In [ ]:
# see link
# https://github.com/Fraud-Detection-Handbook

package ='08-sklearn.tree'
name='Decision Tree'
tuningAndParameters='02-After tuning'

hyperparametersFound = {'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 9, 'min_samples_split': 2}
scalerFound='None'


print('done')

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)

import result as resultMd
reload(resultMd)

import graph as gf
reload(gf)

print('done')

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x_train, x_test, y_train, y_test = train_test_split(dfLearning[predictors], dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

In [ ]:
%%script false

from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

#{'criterion': 'entropy', 'max_depth': 9, 'min_samples_leaf': 7, 'min_samples_split': 32
dic_param={
    'criterion':["gini","entropy"],
    'max_depth': randint(2,12),
    'min_samples_leaf': randint(2,10),
    'min_samples_split': randint(10,20)
}
modelClf = DecisionTreeClassifier(random_state=42)

random_search = RandomizedSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)


#{'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 9, 'min_samples_split': 11}
#0.7697007263257162

In [ ]:
%%script false

from skopt import BayesSearchCV
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from skopt.plots import plot_convergence

# Define model and search for best hyperparameter combination
from skopt import BayesSearchCV, space, plots

clf = DecisionTreeClassifier(random_state=42)

params = {
    'criterion':space.Categorical(["gini","entropy"]),
    'max_depth': space.Integer(2,12),
    'min_samples_leaf': space.Integer(2,10),
    'min_samples_split': space.Integer(10,20)
}

search = BayesSearchCV(clf, params, n_iter = 50, cv = 4, scoring = 'f1',
                       refit = False, verbose = 2)

search.fit(x_train, y_train)

clf.set_params(**search.best_params_)



print("val. score: %s" % search.best_score_)
#print("test score: %s" % search.score(x_test, y_test))
print("best params: %s" % str(search.best_params_))

plot_convergence(search.optimizer_results_[0])
plt.show()

# 10
#val. score: 0.7580348575065415
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 6, 'min_samples_leaf': 9, 'min_samples_split': 13})

# 15
#val. score: 0.7565948397607452
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 10, 'min_samples_split': 20})

#20
#val. score: 0.7603209369000552
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 7, 'min_samples_split': 20})

#25
#val. score: 0.7674618996987418
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 7, 'min_samples_split': 14})

#30
#val. score: 0.7697007263257162
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 12, 'min_samples_leaf': 9, 'min_samples_split': 10})

#40
#val. score: 0.7697007263257162
#best params: OrderedDict({'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 9, 'min_samples_split': 20})


In [ ]:
%%script false
import matplotlib.pyplot as plt
from skopt.plots import plot_convergence

print("val. score: %s" % search.best_score_)
#print("test score: %s" % search.score(x_test, y_test))
print("best params: %s" % str(search.best_params_))

plot_convergence(search.optimizer_results_[0])
plt.show()

In [ ]:
%%script false

from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
from sklearn.model_selection import GridSearchCV

dic_param={
    'criterion':["gini","entropy"],
    'max_depth': [2,3,4,5,6,8,10,11],
    'min_samples_leaf': [2,3,4,5,6,7,8,9,10,11],
    'min_samples_split': [2,3,4,5,6,7,8,9,10,11,12,13]
}
modelClf = DecisionTreeClassifier(random_state=42)

random_search = GridSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)


#{'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 9, 'min_samples_split': 11}
#0.7697007263257162

#{'criterion': 'entropy', 'max_depth': 11, 'min_samples_leaf': 9, 'min_samples_split': 2}
#0.7697007263257162


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from datetime import datetime

modelClf = DecisionTreeClassifier(random_state=42)
parameters=hyperparametersFound

modelClf.set_params(**parameters)
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
learningDurationInS = duration.total_seconds()
print("Duration ",learningDurationInS )
resultMd.update_time_response_result(package, name, tuningAndParameters, learningDurationInS)

predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

f1Learning =f1_score(y_train, predsTrain)
f1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {f1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {f1Test:.3f}")
resultMd.update_learning_test_result(package, name, tuningAndParameters, f1Learning,f1Test)

In [ ]:
gf.show_importance(modelClf, predictors)

In [ ]:
gf.show_prediction_graph(modelClf, x_test,y_test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
# Plot the decision tree
plt.figure(figsize=(20, 10))
plot_tree(modelClf, filled=True, feature_names=x_train.columns, class_names=['Geniune', 'Fraudulent'])
plt.show()

In [ ]:
predsValidation = modelClf.predict(dfValidation[predictors])
f1Validation=f1_score(dfValidation[target], predsValidation)

dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print(f"f1 validation {f1Validation:.3f}")
resultMd.update_performance_result(package, name, tuningAndParameters, f1Validation)
resultMd.update_hyperparameters_result(package, name, tuningAndParameters, hyperparametersFound,scalerFound)


# Summary

In [ ]:
print('Summary')
print(f"{package} {name} {tuningAndParameters}") 
print(f"hyperparameters {hyperparametersFound}") 
print(f"scaler {scalerFound}") 
print('-----------------------------')

print(f"learning duration {learningDurationInS:.2f} s")
print('-----------------------------')
print(f"f1 train      {f1Learning:.3f}")
print(f"f1 test       {f1Test:.3f}")
print(f"f1 validation {f1Validation:.3f}")